# Greater Nanaimo Pollution Control Centre

In [46]:
import datetime as dt
import gsw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import xarray as xr
import PyCO2SYS as pyco2

In [34]:
data = xr.open_dataset("/ocean/dtaneja/MOAD/analysis-dishika/notebooks/WWTP/canada_wwtp_sources.nc")
data

<xarray.Dataset> Size: 520MB
Dimensions:          (y: 898, x: 398, time_counter: 12)
Coordinates:
  * y                (y) int64 7kB 0 1 2 3 4 5 6 ... 891 892 893 894 895 896 897
  * x                (x) int64 3kB 0 1 2 3 4 5 6 ... 391 392 393 394 395 396 397
  * time_counter     (time_counter) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
Data variables: (12/18)
    nav_lat          (y, x) float32 1MB ...
    nav_lon          (y, x) float32 1MB ...
    area             (y, x) float64 3MB ...
    flux             (time_counter, y, x) float64 34MB ...
    temperature      (time_counter, y, x) float64 34MB ...
    NO3              (time_counter, y, x) float64 34MB ...
    ...               ...
    DON              (time_counter, y, x) float64 34MB ...
    bSi              (time_counter, y, x) float64 34MB ...
    oxygen           (time_counter, y, x) float64 34MB ...
    alkalinity       (time_counter, y, x) float64 34MB ...
    DIC              (time_counter, y, x) float64 34MB ...
    turb             (time_counter, y, x) float64 34MB ...

In [35]:
ds = xr.open_dataset("/ocean/cstang/MOAD/analysis-camryn/OAE/DyeTracing/Nanaimo/wastewaterNanaimo_20240601_wNutrients_Adjusted.nc")
nanaimo_mask = (ds["flux"].fillna(0).max(dim="time_counter") > 0)
nanaimo_indices = np.argwhere(nanaimo_mask.values)
j = int(nanaimo_indices[0, 0])
i = int(nanaimo_indices[0, 1])
print(f"Nanaimo grid cell: j={j}, i={i}")

Nanaimo grid cell: j=498, i=215


## Temperature (deg C)

In [ ]:
n_year = 2
temp_2024 = np.array([13.9,13.8,13.9,15.4,17.3,18.9,21.1,21.3,20.2,17.7,15.4,14.4])
temp_2025 = np.array([14.0,13.2,13.7,15.4,17.4,19.4,21.1,21.2,20.7,18.3,16.6,15.2])
temp = (temp_2024+temp_2025)/2

array([13.95, 13.5 , 13.8 , 15.4 , 17.35, 19.15, 21.1 , 21.25, 20.45,
       18.  , 16.  , 14.8 ])

In [29]:
# Conservative temperature 
temp_conservative = gsw.CT_from_pt(0, temp)
print(temp_conservative)

[14.69945758 14.22713223 14.54202257 16.22099806 18.26634952 20.15361365
 22.19746224 22.35465517 21.51625143 18.94793865 16.85043578 15.59146593]


## Flux (m^3/day)

In [26]:
flux_m3d_2024 = np.array([48960,38196,38693,31203,29092,28800,27926,27800,28057,30724,45818,47561])
flux_m3d_2025 = np.array([34669,39806,42319,31997,28995,28910,28292,28707,27850,29704,35841,43326])
flux_m3d = (flux_m3d_2024+flux_m3d_2025)/2
flux_m3d

array([41814.5, 39001. , 40506. , 31600. , 29043.5, 28855. , 28109. ,
       28253.5, 27953.5, 30214. , 40829.5, 45443.5])

In [27]:
mesh_mask_nc = "/ocean/atall/MOAD/grid/mesh_mask_202310b.nc"
ds_mask = xr.open_dataset(mesh_mask_nc)
rho = 1000
seconds_per_day = 86400
cell_area = (ds_mask["e1t"].isel(t=0).values[j, i]* ds_mask["e2t"].isel(t=0).values[j, i])
new_flux = (flux_m3d * rho/ (cell_area * seconds_per_day))
print("New flux:", new_flux)
print("New flux is in kg/m2/S")


New flux: [0.00225015 0.00209875 0.00217974 0.00170048 0.00156291 0.00155277
 0.00151262 0.0015204  0.00150425 0.0016259  0.00219715 0.00244544]
New flux is in kg/m2/S


## Ammonia (mg/L)

In [7]:
ammonia_2024 = np.array([20.4,22.7,15.1,20.0,18.7,16.8,18.2,27.1,33.1,19.6,17.9,15.6])
ammonia_2025 = np.array([12.7,12.8,16.2,12.3,21.3,18.5,19.2,25.7,18.9,21.8,15.7,16.3])
ammonia = (ammonia_2024+ammonia_2025)/2
ammonia

array([16.55, 17.75, 15.65, 16.15, 20.  , 17.65, 18.7 , 26.4 , 26.  ,
       20.7 , 16.8 , 15.95])

In [30]:
# Convert mg/L NH3 to mmol/m³
molar_mass_NH3 = 17.031  # g/mol
ammonia_mmol_m3 = ammonia * 1000 / molar_mass_NH3
print(ammonia_mmol_m3)

[ 971.75738359 1042.21713346  918.91257119  948.27080031 1174.32916446
 1036.34548764 1097.99776877 1550.11449709 1526.6279138  1215.43068522
  986.43649815  936.52750866]


## Alkalinity (mg/L)

In [8]:
alkalinity_2024 = np.array([108,116,82.6,106,96.1,78.1,89.1,137,154,109,104,98.6])
alkalinity_2025 = np.array([73.9,71.6,96.0,64.6,109,103,93.7,121,87,101,81.1,90.5])
alkalinity = (alkalinity_2024+alkalinity_2025)/2
alkalinity

array([ 90.95,  93.8 ,  89.3 ,  85.3 , 102.55,  90.55,  91.4 , 129.  ,
       120.5 , 105.  ,  92.55,  94.55])

In [44]:
# mg/L as CaCO3 to mmol/m³ of alkalinity
alkalinity_mmol_m3 = alkalinity * 1000 / 100.0869 * 2
print(alkalinity_mmol_m3)

[1817.42066145 1874.37117145 1784.44931355 1704.51877319 2049.21922849
 1809.42760741 1826.41284724 2577.75992662 2407.90752836 2098.17668446
 1849.39287759 1889.35814777]


## BOD (mg/L)

In [9]:
bod_2024 = np.array([8.64,6.24,6.22,9.63,10.2,9.84,12.1,10.1,7.13,6.82,6.32,10.4])
bod_2025 = np.array([8.15,8.04,5.68,4.67,7.67,7.54,8.02,7.59,7.42,6.46,6.29,7.05])
bod = (bod_2024+bod_2025)/2
bod

array([ 8.395,  7.14 ,  5.95 ,  7.15 ,  8.935,  8.69 , 10.06 ,  8.845,
        7.275,  6.64 ,  6.305,  8.725])

## PON and DON

In [51]:
# Convert BOD from mg O2/L to mmol O2/m3,then estimate biodegradable organic carbon in mmol C/m3
boc_mmol_m3 = bod * 1000 / 32 * 106 / 138
# Assumed molar carbon-to-nitrogen ratio
carbon_to_nitrogen_ratio = 34
# Assumption: DOC = 0.8 * POC
doc_to_poc_ratio = 0.8
# Split biodegradable organic carbon into particulate and dissolved carbon
poc_mmol_m3 = boc_mmol_m3 / (1 + doc_to_poc_ratio)
doc_mmol_m3 = poc_mmol_m3 * doc_to_poc_ratio
# Convert particulate and dissolved carbon to organic nitrogen
PON = poc_mmol_m3 / carbon_to_nitrogen_ratio
DON = doc_mmol_m3 / carbon_to_nitrogen_ratio

print("Estimated PON in mmol N/m3:")
print(PON)

print("\nEstimated DON in mmol N/m3:")
print(DON)

Estimated PON in mmol N/m3:
[3.29265387 2.80042271 2.33368559 2.80434487 3.50445054 3.40835761
 3.94569362 3.4691511  2.85337187 2.60431467 2.47292229 3.42208517]

Estimated DON in mmol N/m3:
[2.63412309 2.24033816 1.86694847 2.24347589 2.80356043 2.72668609
 3.15655489 2.77532088 2.2826975  2.08345174 1.97833783 2.73766813]


## pH

In [10]:
ph_2024 = np.array([6.93,6.99,6.80,6.95,6.86,6.81,6.86,7.04,7.14,7,6.93,6.87])
ph_2025 = np.array([6.71,6.63,6.80,6.68,6.79,6.84,6.89,7.04,6.97,6.99,6.83,6.90])
ph = (ph_2024+ph_2025)/2
ph

array([6.82 , 6.81 , 6.8  , 6.815, 6.825, 6.825, 6.875, 7.04 , 7.055,
       6.995, 6.88 , 6.885])

## DIC

In [ ]:
# At density = 1000 kg/m3: mmol/m3 is numerically approximately equal to µmol/kg
alkalinity_umol_kg = alkalinity_mmol_m3

kwargs = dict(par1_type=1, par1=alkalinity_umol_kg,    # Total alkalinity
    par2_type=3,par2=ph,                               # pH
    salinity=0.4,
    temperature=temp,
    pressure=0,
    total_silicate=137,       
    total_phosphate=0,           
    opt_pH_scale=4,            
    opt_k_carbonic=8,
    opt_k_bisulfate=1,
    opt_total_borate=1,
)

results = pyco2.sys(**kwargs)
dic_umol_kg = results["dic"]
rho = 1000.0  # kg/m3
dic_mmol_m3 = dic_umol_kg * rho / 1000
print("Monthly DIC in mmol/m3:")
print(dic_mmol_m3)

Monthly DIC in mmol/m³:
[2554.50314023 2659.25516331 2544.57735113 2384.66096831 2820.59417195
 2470.24121864 2402.86650957 3132.03484536 2914.11979133 2625.57855646
 2477.53308591 2538.10689655]


## TSS (mg/L)

In [11]:
tss_2024 = np.array([11.2, 8.05, 7.69, 11.4, 13.2, 9.88, 13.1, 10.5, 8.34, 6.97, 7.05, 12.3])
tss_2025 = np.array([10.6, 8.46, 7.36, 6.63, 16.3, 12.8, 11.6, 7.74, 5.88, 6.73, 8.94, 8.85])
tss = (tss_2024 + tss_2025) / 2
tss

array([10.9  ,  8.255,  7.525,  9.015, 14.75 , 11.34 , 12.35 ,  9.12 ,
        7.11 ,  6.85 ,  7.995, 10.575])

## Turbidity

In [50]:
turbidity_fau = (tss - 7) / 0.89
print("Turbidity in FAU:")
print(turbidity_fau)

Turbidity in FAU:
[ 4.38202247  1.41011236  0.58988764  2.26404494  8.70786517  4.87640449
  6.01123596  2.38202247  0.12359551 -0.16853933  1.11797753  4.01685393]


In [36]:
j = int(nanaimo_indices[0, 0])
i = int(nanaimo_indices[0, 1])
print(f"Nanaimo grid cell: j={j}, i={i}")
no3_values = ds["NO3"].isel(y=j, x=i).values
print(no3_values)

Nanaimo grid cell: j=498, i=215
[322. 322. 322. 322. 322. 322. 322. 322. 322. 322. 322. 322.]


In [37]:
nanoflagelltes_values = ds["nanoflagellates"].isel(y=j, x=i).values
print(nanoflagelltes_values)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [39]:
z1_values = ds["Z1"].isel(y=j, x=i).values
print(z1_values)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [40]:
diatoms_values = ds["diatoms"].isel(y=j, x=i).values
print(diatoms_values)
dSi_values = ds["dSi"].isel(y=j, x=i).values
print(dSi_values)
bSi_values = ds["dSi"].isel(y=j, x=i).values
print(dSi_values)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [41]:
oxygen_values = ds["oxygen"].isel(y=j, x=i).values
print(oxygen_values)

[184. 184. 184. 184. 184. 184. 184. 184. 184. 184. 184. 184.]


In [14]:
gridcoords = 'coordinates_seagrid_SalishSea201702.nc'
coords_file = '/home/sallen/MEOPAR/grid/'+gridcoords
fB = xr.open_dataset(coords_file, decode_times=False)
lat = fB['nav_lat'][:]
lon = fB['nav_lon'][:]
e1t = fB['e1t'][0,:]
e2t = fB['e2t'][0,:]
horz_area = e1t*e2t
fB.close()

In [15]:
print(horz_area)

<xarray.DataArray (y: 898, x: 398)> Size: 3MB
array([[185243.16065032, 190803.27747661, 194671.95335081, ...,
        231244.7650926 , 231250.57782804, 231256.38616842],
       [185143.22065913, 190700.12381496, 194576.64831066, ...,
        231218.44692684, 231224.25621104, 231230.06109609],
       [192573.97235167, 194186.67154445, 196368.25566354, ...,
        231178.55957774, 231184.36291238, 231190.16184242],
       ...,
       [234534.93463225, 234471.10689115, 234011.83013385, ...,
        178285.33125841, 177754.66344186, 177255.24489697],
       [236196.92164773, 236103.23266655, 235423.37323571, ...,
        177713.53480049, 177057.4920366 , 176408.64494444],
       [238326.73957072, 238154.91585215, 237066.18933351, ...,
        177165.37586233, 176356.42614012, 175503.8701075 ]],
      shape=(898, 398))
Coordinates:
    time     float32 4B 9.969e+36
Dimensions without coordinates: y, x
